# Stage 03c — Frozen open-set clustering evaluation

Loads Stage 03a open-set WildFusion artifacts and frozen Stage 03b thresholds.

- Each `_open.npy` already contains **both** `sim_query_query` (qq) and `sim_query_db` (qd / gallery). There are no separate qq/qd files.
- **Gallery animals** (gcn / elephants / bristol / spac): Optuna winners from 03b, with qq clustering + qd known-ID attachment.
- **Jaguar:** gallery-free. Uses frozen label-free defaults only (no Optuna). qd has zero gallery columns.
- Local defaults look under per-model folders so swapping `GLOBAL_DESCRIPTOR` does not overwrite prior runs:
  - matrices: `outputs/stage03/03a/{GLOBAL_DESCRIPTOR}/{dataset}/sim_matrix_…_open.npy`
  - configs: `outputs/stage03/03b/{GLOBAL_DESCRIPTOR}/{dataset}/best_configs_….json`

Cluster metrics on the final predicted labels: ARI, AMI/NMI, homogeneity/completeness/V-measure, BCubed P/R/F1, pairwise P/R/F1. Open-set attachment diagnostics reuse Stage-02 BAKS/BAUS/NormAcc definitions where *accepted* means attached to a gallery identity.


In [5]:
%pip install -q networkx scikit-learn

from collections import defaultdict
import hashlib
import json
import os
import random
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    completeness_score,
    homogeneity_score,
    normalized_mutual_info_score,
    v_measure_score,
)

# ============================================================
# USER KNOBS
# ============================================================
GLOBAL_DESCRIPTOR = "ApexID_BHCA_A_v2"
# Animals with local 03a _open.npy matrices. SPAC still skipped.
DATASET_NAMES = ("bristol_gorillas", "elephants", "gcn_id", "jaguar_antonio")
SPLIT_SEEDS = (42,)

ALLOWED_DATASETS = ("bristol_gorillas", "spac_wild", "elephants", "gcn_id", "jaguar_antonio")
ALLOWED_GLOBAL_DESCRIPTORS = (
    "ApexID_BHCA_A_v2", "ApexID_BHCA_SCA_v2", "ApexID_BHCA_SCSF2_v2", "MiewIDv3", "MegaDescriptor",
)
assert GLOBAL_DESCRIPTOR in ALLOWED_GLOBAL_DESCRIPTORS
if DATASET_NAMES == "all":
    SELECTED_DATASETS = ALLOWED_DATASETS
else:
    SELECTED_DATASETS = tuple(DATASET_NAMES)
    extra = set(SELECTED_DATASETS) - set(ALLOWED_DATASETS)
    if extra:
        raise ValueError(f"Unknown DATASET_NAMES: {sorted(extra)}")
    if not SELECTED_DATASETS:
        raise ValueError("DATASET_NAMES is empty")

# Frozen 03b Optuna winners + Jaguar label-free defaults (same as 03b QUERY_ONLY_DEFAULTS).
FROZEN_CONFIGS = {
    "gcn_id": {
        "top_r": 2, "tau_attach": 0.13, "tau_margin": 0.27, "cw_threshold": 0.32,
        "cw_top_k": 17, "cw_iterations": 22, "cw_k_core_degree": 1,
        "rr_k1": 49, "rr_k2": 7, "rr_lambda": 0.736837942174189, "use_rerank": True,
        "protocol": "gallery_attach_optuna", "config_source": "optuna_stage03_open_split",
        "objective": "0.6*ARI+0.4*V-measure", "objective_value": 0.9597963627032752,
        "validation_ari": 0.9374100658919262, "validation_v_measure": 0.9933758079202986,
        "n_trials_completed": 2000, "split_seed": 42, "known_identity_frac": 0.8,
        "db_frac_within_known": 0.8, "min_images_per_known_identity": 3,
        "dataset": "gcn_id", "global_descriptor": "ApexID_BHCA_A_v2",
    },
    "elephants": {
        "top_r": 2, "tau_attach": 0.42, "tau_margin": 0.59, "cw_threshold": 0.39,
        "cw_top_k": 13, "cw_iterations": 6, "cw_k_core_degree": 1,
        "rr_k1": 41, "rr_k2": 10, "rr_lambda": 0.8766937839074717, "use_rerank": True,
        "protocol": "gallery_attach_optuna", "config_source": "optuna_stage03_open_split",
        "objective": "0.6*ARI+0.4*V-measure", "objective_value": 0.49904094520075165,
        "validation_ari": 0.23850837593616134, "validation_v_measure": 0.8898397990976371,
        "n_trials_completed": 2000, "split_seed": 42, "known_identity_frac": 0.8,
        "db_frac_within_known": 0.8, "min_images_per_known_identity": 3,
        "dataset": "elephants", "global_descriptor": "ApexID_BHCA_A_v2",
    },
    "bristol_gorillas": {
        "top_r": 5, "tau_attach": 0.84, "tau_margin": 0.2, "cw_threshold": 0.13,
        "cw_top_k": 18, "cw_iterations": 15, "cw_k_core_degree": 1,
        "rr_k1": 50, "rr_k2": 13, "rr_lambda": 0.49561469479285564, "use_rerank": True,
        "protocol": "gallery_attach_optuna", "config_source": "optuna_stage03_open_split",
        "objective": "0.6*ARI+0.4*V-measure", "objective_value": 0.28938726066971465,
        "validation_ari": 0.1691367413161989, "validation_v_measure": 0.46976303969998834,
        "n_trials_completed": 2000, "split_seed": 42, "known_identity_frac": 0.8,
        "db_frac_within_known": 0.8, "min_images_per_known_identity": 3,
        "dataset": "bristol_gorillas", "global_descriptor": "ApexID_BHCA_A_v2",
    },
    "jaguar_antonio": {
        "cw_threshold": 0.30, "cw_top_k": 20, "cw_iterations": 20, "cw_k_core_degree": 1,
        "use_rerank": True, "rr_k1": 20, "rr_k2": 6, "rr_lambda": 0.30,
        "protocol": "query_only_defaults", "config_source": "frozen_label_free_defaults",
        "tuning_performed": False, "n_trials_completed": 0, "database_required": False,
        "known_attachment_enabled": False, "cw_seed": 42,
        "dataset": "jaguar_antonio", "global_descriptor": "ApexID_BHCA_A_v2",
    },
}


def protocol_for(dataset_name: str) -> str:
    return "query_only_defaults" if dataset_name == "jaguar_antonio" else "gallery_attach_optuna"


ON_KAGGLE = Path("/kaggle/working").is_dir()
HERE = Path.cwd().resolve()
REPO_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
REPO_CANDIDATES = [REPO_ROOT, HERE, HERE / "notebooks", REPO_ROOT / "notebooks"]


def find_artifact(filename: str, explicit_env: str, dataset_name: str | None = None) -> Path:
    """Prefer outputs/stage03/{03a|03b}/{GLOBAL_DESCRIPTOR}/{dataset}/ so models do not collide."""
    dataset = dataset_name or ""
    is_matrix = filename.startswith("sim_matrix_") and filename.endswith(".npy")
    stage = "03a" if is_matrix else "03b"
    candidates = [Path(os.environ.get(explicit_env, ""))]
    for base in REPO_CANDIDATES:
        staged = base / "outputs" / "stage03" / stage / GLOBAL_DESCRIPTOR
        candidates.extend(
            [
                staged / dataset / filename if dataset else Path(""),
                staged / filename,
                base / filename,
                base / "notebooks" / filename,
                base / "stage03b_configs" / filename,
                base / "notebooks" / "stage03b_configs" / filename,
            ]
        )
    # Shallow Kaggle mounts only — do not walk all of /kaggle/input.
    for parent in (
        Path("/kaggle/working"),
        Path("/kaggle/input/datasets/mohamedbassat"),
        Path("/kaggle/input"),
    ):
        if not parent.is_dir():
            continue
        candidates.append(parent / filename)
        try:
            children = list(parent.iterdir())
        except OSError:
            continue
        for child in children[:40]:
            if child.is_dir():
                candidates.extend(
                    [
                        child / filename,
                        child / GLOBAL_DESCRIPTOR / dataset / filename if dataset else Path(""),
                        child / GLOBAL_DESCRIPTOR / filename,
                        child / stage / GLOBAL_DESCRIPTOR / dataset / filename if dataset else Path(""),
                        child / stage / GLOBAL_DESCRIPTOR / filename,
                        child / "stage03b_configs" / filename,
                        child / "apexid_cvpr_stage03" / stage / GLOBAL_DESCRIPTOR / dataset / filename if dataset else Path(""),
                        child / "apexid_cvpr_stage03" / stage / GLOBAL_DESCRIPTOR / filename,
                    ]
                )
            elif child.name == filename:
                candidates.append(child)
    path = next((c.resolve() for c in candidates if str(c) and c.is_file()), None)
    if path is None:
        raise FileNotFoundError(f"Could not find {filename}")
    return path


def resolve_cluster_labels(cluster_ids, best_names, best_scores, margins, tau_attach, tau_margin):
    votes = defaultdict(lambda: defaultdict(float))
    for index, cluster in enumerate(cluster_ids):
        if best_scores[index] >= tau_attach and margins[index] >= tau_margin:
            votes[int(cluster)][str(best_names[index])] += float(best_scores[index])
    known = {
        cluster: max(identity_votes.items(), key=lambda item: item[1])[0]
        for cluster, identity_votes in votes.items() if identity_votes
    }
    return np.asarray(
        [known.get(int(cluster), f"__new_cluster_{int(cluster)}") for cluster in cluster_ids],
        dtype=str,
    )


def compute_identity_scores(sim_query_db, db_ids, top_r):
    identity_names = list(dict.fromkeys(db_ids.tolist()))
    scores = np.zeros((sim_query_db.shape[0], len(identity_names)), dtype=np.float32)
    for j, identity in enumerate(identity_names):
        cols = np.flatnonzero(db_ids == identity)
        sims = sim_query_db[:, cols]
        r = min(int(top_r), sims.shape[1])
        scores[:, j] = sims.mean(axis=1) if r == sims.shape[1] else np.partition(sims, -r, axis=1)[:, -r:].mean(axis=1)
    return np.asarray(identity_names, dtype=object), scores


def summarize_best_identities(names, scores):
    best_idx = scores.argmax(axis=1)
    best_scores = scores[np.arange(len(scores)), best_idx]
    best_names = names[best_idx]
    if scores.shape[1] >= 2:
        top2 = np.partition(scores, -2, axis=1)[:, -2:]
        second = np.min(top2, axis=1)
    else:
        second = np.zeros(len(scores), dtype=np.float32)
    return best_names, best_scores, second, best_scores - second


def get_topk_neighbors(sim_matrix, top_k):
    n = sim_matrix.shape[0]
    k = min(int(top_k), n - 1)
    if k <= 0:
        return np.empty((n, 0), dtype=np.int64)
    sims = sim_matrix.copy()
    np.fill_diagonal(sims, -np.inf)
    return np.argpartition(sims, -k, axis=1)[:, -k:]


def chinese_whispers(sim_matrix, threshold, iterations, top_k, k_core_degree, seed=42, skip_mutual_knn=False):
    random.seed(seed)
    np.random.seed(seed)
    n = sim_matrix.shape[0]
    adj = sim_matrix.copy()
    np.fill_diagonal(adj, 0)
    adj[adj < threshold] = 0
    graph = nx.from_numpy_array(adj)
    if graph.number_of_edges():
        try:
            backbone = nx.k_core(graph, k=int(k_core_degree))
            if backbone.number_of_nodes() == 0:
                backbone = graph
        except Exception:
            backbone = graph
    else:
        backbone = graph
    backbone_nodes = list(backbone.nodes()) or list(range(n))
    in_backbone = np.zeros(n, dtype=bool)
    in_backbone[backbone_nodes] = True
    valid = (sim_matrix >= threshold) & in_backbone[:, None] & in_backbone[None, :]
    np.fill_diagonal(valid, False)
    if not skip_mutual_knn:
        topk = get_topk_neighbors(sim_matrix, top_k)
        nearest = np.zeros((n, n), dtype=bool)
        if topk.shape[1]:
            nearest[np.arange(n)[:, None], topk] = True
        valid &= nearest & nearest.T
    sources, targets = np.nonzero(valid)
    graph_list = [[] for _ in range(n)]
    for source, target in zip(sources, targets):
        graph_list[int(source)].append((int(target), float(sim_matrix[source, target])))
    labels = list(range(n))
    nodes = backbone_nodes.copy()
    for _ in range(int(iterations)):
        random.shuffle(nodes)
        changes = 0
        for node in nodes:
            weights = defaultdict(float)
            for neighbor, weight in graph_list[node]:
                weights[labels[neighbor]] += weight
            if weights:
                new_label = max(weights.items(), key=lambda item: (item[1], -item[0]))[0]
                changes += int(new_label != labels[node])
                labels[node] = new_label
        if changes == 0:
            break
    return np.unique(labels, return_inverse=True)[1].astype(np.int64)


def re_ranking(sim_matrix, k1=20, k2=6, lambda_value=0.3):
    sim_matrix = np.asarray(sim_matrix, dtype=np.float32)
    n = sim_matrix.shape[0]
    original_dist = np.clip(1.0 - sim_matrix, 0.0, None)
    col_max = np.max(original_dist, axis=0)
    col_max[col_max == 0] = 1.0
    original_dist = (original_dist / col_max).T.astype(np.float32)
    values = np.zeros((n, n), dtype=np.float32)
    initial_rank = np.argsort(original_dist, axis=1).astype(np.int32)
    half_k1 = int(np.around(k1 / 2.0)) + 1
    for i in range(n):
        forward = initial_rank[i, : k1 + 1]
        backward = initial_rank[forward, : k1 + 1]
        reciprocal = forward[np.where(backward == i)[0]]
        expanded = reciprocal.copy()
        for q in reciprocal:
            q_forward = initial_rank[q, :half_k1]
            q_backward = initial_rank[q_forward, :half_k1]
            q_recip = q_forward[np.where(q_backward == q)[0]]
            if len(q_recip) and len(np.intersect1d(q_recip, reciprocal)) > (2.0 / 3.0) * len(q_recip):
                expanded = np.append(expanded, q_recip)
        expanded = np.unique(expanded)
        weight = np.exp(-original_dist[i, expanded])
        if weight.sum() > 0:
            values[i, expanded] = weight / weight.sum()
    if k2 > 1:
        expanded_values = np.zeros_like(values)
        for i in range(n):
            expanded_values[i] = np.mean(values[initial_rank[i, :k2]], axis=0)
        values = expanded_values
    jaccard = np.zeros((n, n), dtype=np.float32)
    inverse = [np.where(values[:, j] != 0)[0] for j in range(n)]
    for i in range(n):
        temp_min = np.zeros(n, dtype=np.float32)
        nonzero = np.where(values[i] != 0)[0]
        for column in nonzero:
            rows = inverse[column]
            temp_min[rows] += np.minimum(values[i, column], values[rows, column])
        jaccard[i] = 1.0 - temp_min / (2.0 - temp_min + 1e-12)
    final_dist = np.clip(jaccard * (1.0 - lambda_value) + original_dist * lambda_value, 0.0, 1.0)
    np.fill_diagonal(final_dist, 0.0)
    return (1.0 - final_dist).astype(np.float32)


local_out = HERE / "outputs" / "stage03" / "03c" / GLOBAL_DESCRIPTOR
if HERE.name == "notebooks":
    local_out = HERE.parent / "outputs" / "stage03" / "03c" / GLOBAL_DESCRIPTOR
OUTPUT_ROOT = Path(os.environ.get(
    "APEXID_STAGE03C_OUTPUT",
    f"/kaggle/working/apexid_cvpr_stage03/03c/{GLOBAL_DESCRIPTOR}" if ON_KAGGLE else str(local_out),
)).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

jobs = []
missing = []
for dataset_name in SELECTED_DATASETS:
    matrix_name = f"sim_matrix_{GLOBAL_DESCRIPTOR}_{dataset_name}_open.npy"
    config_name = f"best_configs_{GLOBAL_DESCRIPTOR}_{dataset_name}.json"
    try:
        matrix_path = find_artifact(matrix_name, "APEXID_STAGE03A_MATRIX", dataset_name)
    except FileNotFoundError as exc:
        missing.append(f"{dataset_name}: {exc}")
        continue
    config_path = None
    try:
        config_path = find_artifact(config_name, "APEXID_STAGE03B_CONFIG", dataset_name)
        config_payload = json.loads(config_path.read_text(encoding="utf-8"))
        if dataset_name not in config_payload:
            raise KeyError(f"{config_name} lacks key {dataset_name}")
        config = config_payload[dataset_name]
    except (FileNotFoundError, KeyError):
        if dataset_name not in FROZEN_CONFIGS:
            missing.append(f"{dataset_name}: no config JSON and no baked FROZEN_CONFIGS entry")
            continue
        config = dict(FROZEN_CONFIGS[dataset_name])
        config_path = OUTPUT_ROOT / config_name
        config_path.write_text(json.dumps({dataset_name: config}, indent=2, sort_keys=True), encoding="utf-8")
        print(f"{dataset_name}: using baked FROZEN_CONFIGS -> wrote {config_path}")
    else:
        # Prefer explicit Optuna JSON when present; still allow jaguar baked defaults.
        config = config_payload[dataset_name]
    jobs.append({
        "dataset": dataset_name,
        "protocol": str(config.get("protocol") or protocol_for(dataset_name)),
        "matrix_path": matrix_path,
        "config_path": config_path,
        "config": config,
    })

if missing:
    raise FileNotFoundError(
        "03c needs each animal's 03a open matrix (qq+qd inside one file). Missing:\n- "
        + "\n- ".join(missing)
    )
print("GLOBAL_DESCRIPTOR:", GLOBAL_DESCRIPTOR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("SELECTED_DATASETS:", [job["dataset"] for job in jobs])
for job in jobs:
    print(
        f"{job['dataset']}: matrix={job['matrix_path']} "
        f"config={job['config_path']} protocol={job['protocol']}"
    )


Note: you may need to restart the kernel to use updated packages.
GLOBAL_DESCRIPTOR: ApexID_BHCA_A_v2
OUTPUT_ROOT: C:\Users\Legion 5\Desktop\ApexID\outputs\stage03\03c\ApexID_BHCA_A_v2
SELECTED_DATASETS: ['bristol_gorillas', 'elephants', 'gcn_id', 'jaguar_antonio']
bristol_gorillas: matrix=C:\Users\Legion 5\Desktop\ApexID\notebooks\sim_matrix_ApexID_BHCA_A_v2_bristol_gorillas_open.npy config=C:\Users\Legion 5\Desktop\ApexID\notebooks\stage03b_configs\best_configs_ApexID_BHCA_A_v2_bristol_gorillas.json protocol=gallery_attach_optuna
elephants: matrix=C:\Users\Legion 5\Desktop\ApexID\notebooks\sim_matrix_ApexID_BHCA_A_v2_elephants_open.npy config=C:\Users\Legion 5\Desktop\ApexID\notebooks\stage03b_configs\best_configs_ApexID_BHCA_A_v2_elephants.json protocol=gallery_attach_optuna
gcn_id: matrix=C:\Users\Legion 5\Desktop\ApexID\notebooks\sim_matrix_ApexID_BHCA_A_v2_gcn_id_open.npy config=C:\Users\Legion 5\Desktop\ApexID\notebooks\stage03b_configs\best_configs_ApexID_BHCA_A_v2_gcn_id.json 

In [6]:
# Load each animal's qq/qd matrices with that animal's frozen 03b thresholds.
def load_animal(job):
    dataset_name = job["dataset"]
    protocol = job["protocol"]
    obj = np.load(job["matrix_path"], allow_pickle=True).item()
    if dataset_name not in obj:
        raise KeyError(f"{job['matrix_path']} keys {list(obj)} do not include {dataset_name}")
    bundle = obj[dataset_name]
    sim_qq = np.asarray(bundle["sim_query_query"], dtype=np.float32)
    sim_qg = np.asarray(bundle.get("sim_query_db", np.zeros((len(sim_qq), 0), dtype=np.float32)), dtype=np.float32)
    query_image_ids = np.asarray(bundle["query_image_ids"]).astype(str)
    db_image_ids = np.asarray(bundle["db_image_ids"]).astype(str)
    y_query = np.asarray(bundle["query_identities"]).astype(str)
    y_db = np.asarray(bundle["db_identities"]).astype(str)
    query_open_role = np.asarray(bundle["query_open_role"]).astype(str)
    if bundle.get("global_descriptor") != GLOBAL_DESCRIPTOR:
        raise ValueError(f"{dataset_name}: 03a global descriptor does not match the 03c knob")
    if bundle.get("dataset") not in {None, dataset_name}:
        raise ValueError(f"{dataset_name}: 03a dataset tag mismatch")
    if sim_qq.shape != (len(query_image_ids), len(query_image_ids)):
        raise ValueError(f"{dataset_name}: query-query shape does not match query_image_ids")
    if sim_qg.shape != (len(query_image_ids), len(db_image_ids)):
        raise ValueError(f"{dataset_name}: query-gallery shape does not match ids")
    if len(set(query_image_ids)) != len(query_image_ids):
        raise ValueError(f"{dataset_name}: query image_ids are not unique")

    if "config" in job and job["config"] is not None:
        config = dict(job["config"])
    else:
        config_payload = json.loads(job["config_path"].read_text(encoding="utf-8"))
        if dataset_name not in config_payload:
            raise KeyError(f"{job['config_path']} does not contain {dataset_name}")
        config = config_payload[dataset_name]
    if config.get("dataset") != dataset_name or config.get("global_descriptor") != GLOBAL_DESCRIPTOR:
        raise ValueError(f"{dataset_name}: frozen config dataset/model does not match")
    if config.get("protocol") != protocol:
        raise ValueError(f"{dataset_name}: frozen config protocol {config.get('protocol')!r} != {protocol!r}")
    required_config = {
        "cw_threshold", "cw_top_k", "cw_iterations", "cw_k_core_degree",
        "use_rerank", "rr_k1", "rr_k2", "rr_lambda",
    }
    if protocol == "gallery_attach_optuna":
        required_config.update({"top_r", "tau_attach", "tau_margin"})
    missing_keys = required_config - set(config)
    if missing_keys:
        raise ValueError(f"{dataset_name}: frozen config is missing {sorted(missing_keys)}")
    if protocol == "query_only_defaults" and (
        config.get("database_required") is not False
        or config.get("known_attachment_enabled") is not False
        or int(config.get("n_trials_completed", -1)) != 0
    ):
        raise ValueError(f"{dataset_name}: query-only config must disable database, attachment, and tuning")
    return {
        "dataset": dataset_name,
        "protocol": protocol,
        "bundle": bundle,
        "config": config,
        "sim_qq": sim_qq,
        "sim_qg": sim_qg,
        "query_image_ids": query_image_ids,
        "db_image_ids": db_image_ids,
        "y_query": y_query,
        "y_db": y_db,
        "query_open_role": query_open_role,
        "matrix_path": job["matrix_path"],
        "config_path": job["config_path"],
    }


loaded_animals = [load_animal(job) for job in jobs]
for animal in loaded_animals:
    print(
        f"{animal['dataset']}: qq={animal['sim_qq'].shape} qd={animal['sim_qg'].shape} "
        f"protocol={animal['protocol']}"
    )
    print(json.dumps(animal["config"], indent=2, sort_keys=True))


bristol_gorillas: qq=(1789, 1789) qd=(1789, 3535) protocol=gallery_attach_optuna
{
  "config_source": "optuna_stage03_open_split",
  "cw_iterations": 15,
  "cw_k_core_degree": 1,
  "cw_threshold": 0.13,
  "cw_top_k": 18,
  "dataset": "bristol_gorillas",
  "db_frac_within_known": 0.8,
  "global_descriptor": "ApexID_BHCA_A_v2",
  "known_identity_frac": 0.8,
  "min_images_per_known_identity": 3,
  "n_trials_completed": 2000,
  "objective": "0.6*ARI+0.4*V-measure",
  "objective_value": 0.28938726066971465,
  "protocol": "gallery_attach_optuna",
  "rr_k1": 50,
  "rr_k2": 13,
  "rr_lambda": 0.49561469479285564,
  "split_seed": 42,
  "tau_attach": 0.84,
  "tau_margin": 0.2,
  "top_r": 5,
  "use_rerank": true,
  "validation_ari": 0.1691367413161989,
  "validation_v_measure": 0.46976303969998834
}
elephants: qq=(821, 821) qd=(821, 1230) protocol=gallery_attach_optuna
{
  "config_source": "optuna_stage03_open_split",
  "cw_iterations": 6,
  "cw_k_core_degree": 1,
  "cw_threshold": 0.39,
  "cw_to

In [7]:
# Frozen evaluation: each animal uses only its own 03b thresholds.
def _balanced_mean(labels, values):
    """Identity-balanced mean used by Stage 02 BAKS/BAUS."""
    labels_array = np.asarray(labels).astype(str)
    values_array = np.asarray(values, dtype=float)
    if not len(values_array):
        return float("nan")
    return float(np.mean([values_array[labels_array == identity].mean() for identity in np.unique(labels_array)]))


def bcubed_scores(y_true, y_pred):
    y_true = np.asarray(y_true).astype(str)
    y_pred = np.asarray(y_pred).astype(str)
    true_groups = {label: np.flatnonzero(y_true == label) for label in np.unique(y_true)}
    pred_groups = {label: np.flatnonzero(y_pred == label) for label in np.unique(y_pred)}
    n = len(y_true)
    if n == 0:
        return 0.0, 0.0, 0.0
    precision = 0.0
    recall = 0.0
    for index in range(n):
        true_members = true_groups[y_true[index]]
        pred_members = pred_groups[y_pred[index]]
        overlap = len(np.intersect1d(true_members, pred_members, assume_unique=False))
        precision += overlap / len(pred_members)
        recall += overlap / len(true_members)
    precision /= n
    recall /= n
    f1 = 0.0 if precision + recall == 0 else 2.0 * precision * recall / (precision + recall)
    return float(precision), float(recall), float(f1)


def pairwise_cluster_scores(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    if n < 2:
        return 0.0, 0.0, 0.0
    true_same = pred_same = both_same = 0
    for i in range(n):
        for j in range(i + 1, n):
            t = y_true[i] == y_true[j]
            p = y_pred[i] == y_pred[j]
            true_same += int(t)
            pred_same += int(p)
            both_same += int(t and p)
    precision = 0.0 if pred_same == 0 else both_same / pred_same
    recall = 0.0 if true_same == 0 else both_same / true_same
    f1 = 0.0 if precision + recall == 0 else 2.0 * precision * recall / (precision + recall)
    return float(precision), float(recall), float(f1)


def clustering_metrics(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true).astype(str)
    y_pred = np.asarray(y_pred).astype(str)
    b_p, b_r, b_f = bcubed_scores(y_true, y_pred)
    p_p, p_r, p_f = pairwise_cluster_scores(y_true, y_pred)
    return {
        f"{prefix}ARI": float(adjusted_rand_score(y_true, y_pred)),
        f"{prefix}AMI": float(adjusted_mutual_info_score(y_true, y_pred)),
        f"{prefix}NMI": float(normalized_mutual_info_score(y_true, y_pred)),
        f"{prefix}V_measure": float(v_measure_score(y_true, y_pred)),
        f"{prefix}homogeneity": float(homogeneity_score(y_true, y_pred)),
        f"{prefix}completeness": float(completeness_score(y_true, y_pred)),
        f"{prefix}BCubed_precision": b_p,
        f"{prefix}BCubed_recall": b_r,
        f"{prefix}BCubed_F1": b_f,
        f"{prefix}pairwise_precision": p_p,
        f"{prefix}pairwise_recall": p_r,
        f"{prefix}pairwise_F1": p_f,
        f"{prefix}n_true_identities": int(pd.Series(y_true).nunique()),
        f"{prefix}n_pred_clusters": int(pd.Series(y_pred).nunique()),
    }


def attachment_open_set_metrics(y_true, y_pred, known_mask, novel_mask, gallery_identities):
    """Stage-02 BAKS/BAUS/NormAcc with 'accepted' = attached to a gallery identity."""
    y_true = np.asarray(y_true).astype(str)
    y_pred = np.asarray(y_pred).astype(str)
    known_mask = np.asarray(known_mask, dtype=bool)
    novel_mask = np.asarray(novel_mask, dtype=bool)
    gallery = set(map(str, gallery_identities))
    accepted = np.asarray([label in gallery for label in y_pred], dtype=bool)
    if known_mask.any():
        known_correct = (y_pred[known_mask] == y_true[known_mask])
        baks = _balanced_mean(y_true[known_mask], (accepted[known_mask] & known_correct).astype(float))
        known_to_new = float(np.mean(~accepted[known_mask]))
    else:
        baks = float("nan")
        known_to_new = float("nan")
    if novel_mask.any():
        baus = _balanced_mean(y_true[novel_mask], (~accepted[novel_mask]).astype(float))
        novel_false_attach = float(np.mean(accepted[novel_mask]))
    else:
        baus = float("nan")
        novel_false_attach = float("nan")
    if np.isfinite(baks) and np.isfinite(baus):
        import math
        normacc = float(math.sqrt(max(baks, 0.0) * max(baus, 0.0)))
    else:
        normacc = float("nan")
    return {
        "BAKS": baks,
        "BAUS": baus,
        "NormAcc": normacc,
        "known_attachment_accuracy": float(np.mean(y_pred[known_mask] == y_true[known_mask])) if known_mask.any() else float("nan"),
        "known_to_new_rate": known_to_new,
        "novel_false_attach_rate": novel_false_attach,
    }


def cluster_query_matrix(sim_qq_matrix, config):
    use_rerank = bool(config["use_rerank"])
    sim_for_cw = re_ranking(
        sim_qq_matrix,
        k1=int(config["rr_k1"]),
        k2=int(config["rr_k2"]),
        lambda_value=float(config["rr_lambda"]),
    ) if use_rerank else sim_qq_matrix
    return chinese_whispers(
        sim_for_cw,
        threshold=float(config["cw_threshold"]),
        iterations=int(config["cw_iterations"]),
        top_k=int(config["cw_top_k"]),
        k_core_degree=int(config["cw_k_core_degree"]),
        seed=int(config.get("cw_seed", 42)),
        skip_mutual_knn=use_rerank,
    )


def evaluate_animal(animal):
    dataset_name = animal["dataset"]
    protocol = animal["protocol"]
    config = animal["config"]
    sim_qq = animal["sim_qq"]
    query_image_ids = animal["query_image_ids"]
    y_query = animal["y_query"]
    if protocol == "query_only_defaults":
        cluster_ids = cluster_query_matrix(sim_qq, config)
        predicted = np.asarray(
            [f"cluster_{dataset_name}_new_{int(cluster)}" for cluster in cluster_ids],
            dtype=str,
        )
        cluster_metrics = clustering_metrics(y_query, predicted)
        # All-novel pool: BAKS undefined; BAUS=1 if nothing is treated as a gallery ID.
        open_metrics = attachment_open_set_metrics(
            y_query, predicted,
            known_mask=np.zeros(len(y_query), dtype=bool),
            novel_mask=np.ones(len(y_query), dtype=bool),
            gallery_identities=[],
        )
        result = {
            "global_descriptor": GLOBAL_DESCRIPTOR,
            "dataset": dataset_name,
            "protocol": protocol,
            "split_seed": "full_pool",
            **cluster_metrics,
            "novel_only_ARI": cluster_metrics["ARI"],
            "novel_only_AMI": cluster_metrics["AMI"],
            "novel_only_V_measure": cluster_metrics["V_measure"],
            "novel_only_BCubed_F1": cluster_metrics["BCubed_F1"],
            **open_metrics,
            "queries": len(query_image_ids),
            "gallery_images": 0,
            "query_known_images": 0,
            "query_novel_images": len(query_image_ids),
            "novel_identities": cluster_metrics["n_true_identities"],
            "predicted_clusters": cluster_metrics["n_pred_clusters"],
            "config_source": config.get("config_source"),
            "cw_threshold": float(config["cw_threshold"]),
            "cw_top_k": int(config["cw_top_k"]),
            "tau_attach": float("nan"),
            "tau_margin": float("nan"),
        }
        n_q = len(query_image_ids)
        details = pd.DataFrame({
            "global_descriptor": [GLOBAL_DESCRIPTOR] * n_q,
            "dataset": [dataset_name] * n_q,
            "protocol": [protocol] * n_q,
            "split_seed": ["full_pool"] * n_q,
            "image_id": query_image_ids,
            "open_role": ["query_novel"] * n_q,
            "true_identity": y_query,
            "raw_cluster_id": cluster_ids,
            "predicted_label": predicted,
            "best_known_identity": [""] * n_q,
            "best_known_score": np.full(n_q, np.nan, dtype=np.float64),
            "second_best_score": np.full(n_q, np.nan, dtype=np.float64),
            "known_margin": np.full(n_q, np.nan, dtype=np.float64),
        })
        print(
            f"{dataset_name} full_pool: ARI={result['ARI']:.4f} AMI={result['AMI']:.4f} "
            f"V={result['V_measure']:.4f} BCubedF1={result['BCubed_F1']:.4f} "
            f"BAUS={result['BAUS']:.4f} clusters={result['predicted_clusters']}"
        )
        return result, details

    query_open_role = animal["query_open_role"]
    db_image_ids = animal["db_image_ids"]
    y_db = animal["y_db"]
    sim_qg = animal["sim_qg"]
    known_mask = query_open_role == "query_known"
    novel_mask = query_open_role == "query_novel"
    if not known_mask.any() or not novel_mask.any():
        raise RuntimeError(f"{dataset_name}: open split must contain known and novel queries")
    if set(query_image_ids) & set(db_image_ids):
        raise RuntimeError(f"{dataset_name}: gallery/query overlap")
    novel_identities = set(y_query[novel_mask])
    if novel_identities & set(y_db):
        raise RuntimeError(f"{dataset_name}: novel identities occur in the gallery")

    identity_names, identity_scores = compute_identity_scores(sim_qg, y_db, top_r=int(config["top_r"]))
    best_names, best_scores, second_scores, margins = summarize_best_identities(identity_names, identity_scores)
    cluster_ids = cluster_query_matrix(sim_qq, config)
    predicted = resolve_cluster_labels(
        cluster_ids, best_names, best_scores, margins,
        tau_attach=float(config["tau_attach"]),
        tau_margin=float(config["tau_margin"]),
    )
    cluster_metrics = clustering_metrics(y_query, predicted)
    novel_metrics = clustering_metrics(y_query[novel_mask], predicted[novel_mask], prefix="novel_only_")
    open_metrics = attachment_open_set_metrics(y_query, predicted, known_mask, novel_mask, y_db)
    split_seed = animal["bundle"].get("split_seed") or (SPLIT_SEEDS[0] if SPLIT_SEEDS else 42)
    result = {
        "global_descriptor": GLOBAL_DESCRIPTOR,
        "dataset": dataset_name,
        "protocol": protocol,
        "split_seed": split_seed,
        **cluster_metrics,
        **novel_metrics,
        **open_metrics,
        "queries": len(query_image_ids),
        "gallery_images": len(db_image_ids),
        "query_known_images": int(known_mask.sum()),
        "query_novel_images": int(novel_mask.sum()),
        "novel_identities": len(novel_identities),
        "predicted_clusters": cluster_metrics["n_pred_clusters"],
        "config_source": config.get("config_source"),
        "cw_threshold": config["cw_threshold"],
        "cw_top_k": config["cw_top_k"],
        "tau_attach": config["tau_attach"],
        "tau_margin": config["tau_margin"],
    }
    details = pd.DataFrame({
        "global_descriptor": GLOBAL_DESCRIPTOR,
        "dataset": dataset_name,
        "protocol": protocol,
        "split_seed": split_seed,
        "image_id": query_image_ids,
        "open_role": query_open_role,
        "true_identity": y_query,
        "raw_cluster_id": cluster_ids,
        "predicted_label": predicted,
        "best_known_identity": best_names.astype(str),
        "best_known_score": best_scores,
        "second_best_score": second_scores,
        "known_margin": margins,
    })
    print(
        f"{dataset_name} seed={split_seed}: ARI={result['ARI']:.4f} AMI={result['AMI']:.4f} "
        f"BCubedF1={result['BCubed_F1']:.4f} novel_ARI={result['novel_only_ARI']:.4f} "
        f"BAKS={result['BAKS']:.4f} BAUS={result['BAUS']:.4f} NormAcc={result['NormAcc']:.4f} "
        f"clusters={result['predicted_clusters']}"
    )
    return result, details


result_rows = []
detail_frames = []
for animal in loaded_animals:
    result, details = evaluate_animal(animal)
    if animal["protocol"] == "query_only_defaults" and (
        int(result["gallery_images"]) != 0
        or int(result["queries"]) != len(animal["query_image_ids"])
    ):
        raise RuntimeError(f"{animal['dataset']}: Jaguar protocol did not use every query image with zero gallery")
    result_rows.append(result)
    detail_frames.append(details)

results = pd.DataFrame(result_rows)
detail_columns = [
    "global_descriptor", "dataset", "protocol", "split_seed", "image_id", "open_role",
    "true_identity", "raw_cluster_id", "predicted_label", "best_known_identity",
    "best_known_score", "second_best_score", "known_margin",
]
aligned = []
for frame in detail_frames:
    piece = frame.copy()
    for column in detail_columns:
        if column not in piece.columns:
            if column in {"best_known_score", "second_best_score", "known_margin"}:
                piece[column] = np.nan
            else:
                piece[column] = ""
    aligned.append(piece[detail_columns])
details = pd.concat(aligned, ignore_index=True)
if set(results["dataset"]) != {job["dataset"] for job in jobs}:
    raise RuntimeError("03c did not produce one result row per selected animal")

finite_metric_cols = [
    "ARI", "AMI", "NMI", "V_measure", "homogeneity", "completeness",
    "BCubed_precision", "BCubed_recall", "BCubed_F1",
    "pairwise_precision", "pairwise_recall", "pairwise_F1", "BAUS",
]
for column in finite_metric_cols:
    if column not in results.columns:
        raise RuntimeError(f"missing metric column {column}")
    if not np.isfinite(results[column]).all():
        raise RuntimeError(f"03c produced non-finite {column}")
gallery_rows = results.loc[results["protocol"].eq("gallery_attach_optuna")]
if len(gallery_rows):
    for column in ("BAKS", "NormAcc", "known_attachment_accuracy"):
        if not np.isfinite(gallery_rows[column]).all():
            raise RuntimeError(f"gallery animals must have finite {column}")
jaguar_rows = results.loc[results["protocol"].eq("query_only_defaults")]
if len(jaguar_rows) and jaguar_rows["BAKS"].notna().any():
    raise RuntimeError("query-only animals must leave BAKS as NaN")
print("Finite cluster + BAUS metrics OK. Jaguar BAKS NaN is expected (no known queries).")

results_path = OUTPUT_ROOT / f"discovery_ari_results_{GLOBAL_DESCRIPTOR}.csv"
details_path = OUTPUT_ROOT / f"discovery_predictions_{GLOBAL_DESCRIPTOR}.csv"
results.to_csv(results_path, index=False)
details.to_csv(details_path, index=False)
for dataset_name, group in results.groupby("dataset"):
    group.to_csv(OUTPUT_ROOT / f"discovery_ari_results_{GLOBAL_DESCRIPTOR}_{dataset_name}.csv", index=False)
summary_cols = [
    "dataset", "protocol", "ARI", "AMI", "NMI", "V_measure", "homogeneity", "completeness",
    "BCubed_F1", "pairwise_F1", "novel_only_ARI", "novel_only_BCubed_F1",
    "BAKS", "BAUS", "NormAcc", "known_attachment_accuracy",
    "known_to_new_rate", "novel_false_attach_rate",
    "n_true_identities", "predicted_clusters",
]
print(results[summary_cols].to_string(index=False))
print("Mean ARI:", float(results["ARI"].mean()))
print("Mean AMI:", float(results["AMI"].mean()))
print("Mean BCubed F1:", float(results["BCubed_F1"].mean()))
print("Mean BAUS:", float(results["BAUS"].mean()))
if gallery_rows is not None and len(gallery_rows):
    print("Mean gallery BAKS:", float(gallery_rows["BAKS"].mean()))
    print("Mean gallery NormAcc:", float(gallery_rows["NormAcc"].mean()))
print("Full metrics CSV:", results_path)


bristol_gorillas seed=42: ARI=0.1691 AMI=0.4489 BCubedF1=0.3063 novel_ARI=0.0000 BAKS=0.0000 BAUS=1.0000 NormAcc=0.0000 clusters=57
elephants seed=42: ARI=0.2385 AMI=0.3621 BCubedF1=0.5955 novel_ARI=0.2373 BAKS=0.0050 BAUS=1.0000 NormAcc=0.0707 clusters=582
gcn_id seed=42: ARI=0.9374 AMI=0.9684 BCubedF1=0.9777 novel_ARI=0.9402 BAKS=0.9398 BAUS=0.9756 NormAcc=0.9575 clusters=211
jaguar_antonio full_pool: ARI=0.3762 AMI=0.6602 V=0.7803 BCubedF1=0.4651 BAUS=1.0000 clusters=194
Finite cluster + BAUS metrics OK. Jaguar BAKS NaN is expected (no known queries).
         dataset              protocol      ARI      AMI      NMI  V_measure  homogeneity  completeness  BCubed_F1  pairwise_F1  novel_only_ARI  novel_only_BCubed_F1     BAKS    BAUS  NormAcc  known_attachment_accuracy  known_to_new_rate  novel_false_attach_rate  n_true_identities  predicted_clusters
bristol_gorillas gallery_attach_optuna 0.169137 0.448857 0.469763   0.469763     0.716209      0.349501   0.306314     0.242354        0.

C:\Users\Legion 5\AppData\Local\Temp\ipykernel_30024\2902949900.py:296: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  details = pd.concat(aligned, ignore_index=True)


In [8]:
# Reproducibility manifest and completion checks.
manifest = {
    "stage": "03c_frozen_open_set_clustering",
    "status": "passed",
    "datasets": list(results["dataset"]),
    "global_descriptor": GLOBAL_DESCRIPTOR,
    "per_animal_thresholds": True,
    "animalclef_repository": "https://github.com/MIA-AI-Team/AnimalCLEF26.git",
    "animalclef_commit": "inlined_from_AnimalCLEF26_cd3d015",
    "threshold_fitting_in_stage03c": False,
    "feature_extraction_in_stage03c": False,
    "animals": {
        animal["dataset"]: {
            "protocol": animal["protocol"],
            "source_matrix": str(animal["matrix_path"]),
            "source_matrix_sha256": hashlib.sha256(animal["matrix_path"].read_bytes()).hexdigest(),
            "source_config": str(animal["config_path"]),
            "source_config_sha256": hashlib.sha256(animal["config_path"].read_bytes()).hexdigest(),
            "config_source": animal["config"].get("config_source"),
            "cw_threshold": animal["config"]["cw_threshold"],
            "cw_top_k": animal["config"]["cw_top_k"],
            "use_rerank": animal["config"]["use_rerank"],
        }
        for animal in loaded_animals
    },
    "mean_ari": float(results["ARI"].mean()),
    "std_ari": float(results["ARI"].std(ddof=1)) if len(results) > 1 else 0.0,
    "mean_novel_only_ari": float(results["novel_only_ARI"].mean()),
    "mean_known_attachment_accuracy": (
        None if results["known_attachment_accuracy"].isna().all()
        else float(results.loc[results["known_attachment_accuracy"].notna(), "known_attachment_accuracy"].mean())
    ),
    "results_file": results_path.name,
    "predictions_file": details_path.name,
}
(OUTPUT_ROOT / "stage03c_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(manifest, indent=2, sort_keys=True))
print(f"\nCOMPLETE: save {OUTPUT_ROOT} as the {GLOBAL_DESCRIPTOR} ARI table across {list(results['dataset'])}.")


{
  "animalclef_commit": "inlined_from_AnimalCLEF26_cd3d015",
  "animalclef_repository": "https://github.com/MIA-AI-Team/AnimalCLEF26.git",
  "animals": {
    "bristol_gorillas": {
      "config_source": "optuna_stage03_open_split",
      "cw_threshold": 0.13,
      "cw_top_k": 18,
      "protocol": "gallery_attach_optuna",
      "source_config": "C:\\Users\\Legion 5\\Desktop\\ApexID\\notebooks\\stage03b_configs\\best_configs_ApexID_BHCA_A_v2_bristol_gorillas.json",
      "source_config_sha256": "fec4444ba4eed95cc366a7fda03121bc49f043b66fa63116806eb0266d446282",
      "source_matrix": "C:\\Users\\Legion 5\\Desktop\\ApexID\\notebooks\\sim_matrix_ApexID_BHCA_A_v2_bristol_gorillas_open.npy",
      "source_matrix_sha256": "d0847ef40646a1b78345facb576c323767214dd08388a038bc8fc34ab98573e8",
      "use_rerank": true
    },
    "elephants": {
      "config_source": "optuna_stage03_open_split",
      "cw_threshold": 0.39,
      "cw_top_k": 13,
      "protocol": "gallery_attach_optuna",
      "s